<a href="https://colab.research.google.com/github/immaculatemuli/Natural_Language_Processing/blob/main/Week_3/Week_3_Mandatory_Tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

WEEK 3: HIDDEN MARKOV MODELS AND SEQUENCE LABELING

1.Sequence Labeling Code

2.Hidden Markov Model Output

3.Named Entity Recognition Example

4.Sentence Labeling Results

5.Dataset Used for Training or Testing

In [ ]:
#
# DATASET: CBK Annual Report & Financial Statements 2024/25
#
# Step 1: Download the PDF from the official CBK website:
#         https://www.centralbank.go.ke/reports/
#         File name: 1084981846_2025 Annual Report.pdf
#
# Step 2: Upload the PDF to Google Colab using one of these methods:
#
#           Upload directly each session:
#         from google.colab import files
#         files.upload()    ← click and select the PDF
#
#
# Step 3: Update the pdf_path variable below to match your file location


In [ ]:
# 1 — Sequence Labeling Code
# Dataset: CBK Annual Report & Financial Statements 2024/25


cbk_text = ''
with open(pdf_path, 'rb') as f:
    reader = PyPDF2.PdfReader(f)
    # Pages 3–5 = Global Economy, Pages 28–29 = Banking Sector
    for page_num in list(range(3, 6)) + list(range(28, 30)):
        page_text = reader.pages[page_num].extract_text()
        if page_text:
            cbk_text += page_text + ' '

print(f"CBK Report loaded: {len(cbk_text):,} characters\n")

# ── Sequence Labeling — label every word in 5 CBK sentences ──
cbk_sentences = [
    "The Monetary Policy Committee lowered the Central Bank Rate to 9.75 percent.",
    "Governor Dr. Kamau Thugge presented the Annual Report in Nairobi.",
    "Kenya met the EAC criteria on headline inflation and gross reserves.",
    "JPMorgan Chase Bank was granted authority to establish an office in Kenya.",
    "The CBK foreign exchange reserves stood at USD 11082 million in June 2025.",
]

tag_meanings = {
    'NN':'Noun',        'NNS':'Noun(plural)',   'NNP':'Proper Noun',
    'VBD':'Verb(past)', 'VBZ':'Verb(present)',  'VBN':'Verb(past part)',
    'JJ':'Adjective',   'RB':'Adverb',          'DT':'Determiner',
    'IN':'Preposition', 'CD':'Number',          'TO':'To',
    'CC':'Conjunction', 'PRP':'Pronoun',        'MD':'Modal',
}

print("=" * 60)
print("  1 — SEQUENCE LABELING CODE")
print("  Dataset: CBK Annual Report & Financial Statements 2024/25")
print("=" * 60)

for i, sent in enumerate(cbk_sentences, 1):
    tokens = word_tokenize(sent)
    tagged = nltk.pos_tag(tokens)
    print(f"\nSentence {i}: {sent}")
    print(f"  {'Word':<25} {'Tag':<10} {'Meaning'}")
    print(f"  {'-'*50}")
    for word, tag in tagged:
        meaning = tag_meanings.get(tag, tag)
        print(f"  {word:<25} {tag:<10} {meaning}")

print("\n" + "=" * 60)
print("Sequence labeling assigns a label to EVERY word in order.")
print("This is the foundation of HMMs and NER.")

CBK Report loaded: 7,359 characters

  1 — SEQUENCE LABELING CODE
  Dataset: CBK Annual Report & Financial Statements 2024/25

Sentence 1: The Monetary Policy Committee lowered the Central Bank Rate to 9.75 percent.
  Word                      Tag        Meaning
  --------------------------------------------------
  The                       DT         Determiner
  Monetary                  NNP        Proper Noun
  Policy                    NNP        Proper Noun
  Committee                 NNP        Proper Noun
  lowered                   VBD        Verb(past)
  the                       DT         Determiner
  Central                   NNP        Proper Noun
  Bank                      NNP        Proper Noun
  Rate                      NNP        Proper Noun
  to                        TO         To
  9.75                      CD         Number
  percent                   NN         Noun
  .                         .          .

Sentence 2: Governor Dr. Kamau Thugge presented the An

In [ ]:
# 2 — Hidden Markov Model Output
# Demonstrating HMM states, observations, transition and emission
# probabilities using CBK Annual Report language

print("=" * 60)
print("  2 — HIDDEN MARKOV MODEL OUTPUT")
print("  Dataset: CBK Annual Report & Financial Statements 2024/25")
print("=" * 60)

#  HMM Components

# Hidden States = POS tags (what the model predicts)
states = ['DT', 'JJ', 'NN', 'VBD', 'IN', 'NNP', 'CD']

# Transition probabilities
# Given current state, probability of moving to next state
transition = {
    'DT':  {'JJ': 0.35, 'NN': 0.55, 'NNP': 0.10},
    'JJ':  {'NN': 0.80, 'JJ': 0.15, 'NNP': 0.05},
    'NN':  {'VBD': 0.45, 'IN': 0.35, 'NN': 0.20},
    'VBD': {'DT': 0.40, 'IN': 0.30, 'CD': 0.20, 'NN': 0.10},
    'IN':  {'DT': 0.50, 'NNP': 0.30, 'CD': 0.20},
    'NNP': {'NNP': 0.30, 'VBD': 0.35, 'IN': 0.35},
    'CD':  {'NN': 0.40, 'IN': 0.35, 'VBD': 0.25},
}

# Emission probabilities
# Given a state, probability of producing a specific CBK word
emission = {
    'DT':  {'the': 0.60, 'a': 0.30, 'its': 0.10},
    'JJ':  {'monetary': 0.30, 'stable': 0.25, 'financial': 0.25, 'overall': 0.20},
    'NN':  {'policy': 0.30, 'sector': 0.25, 'rate': 0.25, 'inflation': 0.20},
    'VBD': {'lowered': 0.35, 'declined': 0.35, 'remained': 0.30},
    'IN':  {'to': 0.35, 'by': 0.30, 'from': 0.20, 'in': 0.15},
    'NNP': {'kenya': 0.30, 'cbk': 0.30, 'nairobi': 0.20, 'mpc': 0.20},
    'CD':  {'9.75': 0.25, '3.8': 0.25, '2025': 0.25, '13.00': 0.25},
}

# ── Simulate sequence: "the monetary policy declined to 9.75" ─
sequence = [
    ('DT',  'the'),
    ('JJ',  'monetary'),
    ('NN',  'policy'),
    ('VBD', 'declined'),
    ('IN',  'to'),
    ('CD',  '9.75'),
]

print("\nHMM Sequence: 'the monetary policy declined to 9.75'")
print("(from CBK Annual Report — CBR rate change)\n")
print(f"  {'Word':<15} {'State':<8} {'Emit P':<12} {'Transition to Next'}")
print(f"  {'-'*55}")

for i, (tag, word) in enumerate(sequence):
    emit_p = emission.get(tag, {}).get(word, 0.01)
    if i < len(sequence) - 1:
        next_tag = sequence[i + 1][0]
        trans_p  = transition.get(tag, {}).get(next_tag, 0.01)
        print(f"  {word:<15} {tag:<8} {emit_p:<12.2f} → {next_tag} (p={trans_p:.2f})")
    else:
        print(f"  {word:<15} {tag:<8} {emit_p:<12.2f} → END")

print("\n" + "=" * 60)
print("HMM CONCEPTS DEMONSTRATED:")
print("  States       = hidden POS tags (NN, VBD, JJ...)")
print("  Observations = actual words seen in CBK report")
print("  Emit P       = probability this state produces this word")
print("  Transition   = probability of moving to the next state")
print("=" * 60)

  2 — HIDDEN MARKOV MODEL OUTPUT
  Dataset: CBK Annual Report & Financial Statements 2024/25

HMM Sequence: 'the monetary policy declined to 9.75'
(from CBK Annual Report — CBR rate change)

  Word            State    Emit P       Transition to Next
  -------------------------------------------------------
  the             DT       0.60         → JJ (p=0.35)
  monetary        JJ       0.30         → NN (p=0.80)
  policy          NN       0.30         → VBD (p=0.45)
  declined        VBD      0.35         → IN (p=0.30)
  to              IN       0.35         → CD (p=0.20)
  9.75            CD       0.25         → END

HMM CONCEPTS DEMONSTRATED:
  States       = hidden POS tags (NN, VBD, JJ...)
  Observations = actual words seen in CBK report
  Emit P       = probability this state produces this word
  Transition   = probability of moving to the next state


In [ ]:
# 3 — Named Entity Recognition (NER) Example
# Using spaCy on the CBK Annual Report 2024/25

!pip install spacy --quiet
!python -m spacy download en_core_web_sm --quiet

import spacy
from collections import defaultdict

nlp = spacy.load('en_core_web_sm')

#  Real paragraph from CBK Annual Report 2024/25
cbk_ner_text = """
The Central Bank of Kenya Annual Report 2024/25 was presented by
Governor Dr. Kamau Thugge to the Cabinet Secretary of the National
Treasury in Nairobi. The Monetary Policy Committee lowered the Central
Bank Rate from 13.00 percent in June 2024 to 9.75 percent by June 2025.
JPMorgan Chase Bank of the United States was granted authority to
establish a Representative Office in Kenya on October 14, 2024.
The CBK hosted the 28th MAC meeting in Mombasa on May 9, 2025.
The IMF and World Bank praised Kenya's monetary policy framework.
CBK foreign exchange reserves stood at USD 11082 million at June 2025.
Overall inflation declined to 3.8 percent in June 2025 from 4.6 percent.
"""

doc = nlp(cbk_ner_text)

# Group entities by label
entities = defaultdict(list)
for ent in doc.ents:
    entities[ent.label_].append(ent.text.strip())

label_names = {
    'PERSON':   'People / Persons',
    'ORG':      'Organisations',
    'GPE':      'Countries / Cities',
    'DATE':     'Dates',
    'PERCENT':  'Percentages',
    'MONEY':    'Money Values',
    'CARDINAL': 'Numbers',
}

print("=" * 60)
print(" 3 — NAMED ENTITY RECOGNITION EXAMPLE")
print("  Dataset: CBK Annual Report & Financial Statements 2024/25")
print("=" * 60)
print("\nEntities automatically extracted — no manual reading needed:\n")

for label in ['PERSON', 'ORG', 'GPE', 'DATE', 'PERCENT', 'MONEY', 'CARDINAL']:
    ents = entities.get(label, [])
    if ents:
        unique_ents = list(dict.fromkeys(ents))
        name = label_names.get(label, label)
        print(f"{name} [{label}]:")
        for e in unique_ents:
            print(f"    → {e}")
        print()

print("=" * 60)
print("NER automatically found people, organisations, locations,")
print("dates and percentages from the CBK report in seconds.")
print("A human analyst would take much longer to do the same.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
 3 — NAMED ENTITY RECOGNITION EXAMPLE
  Dataset: CBK Annual Report & Financial Statements 2024/25

Entities automatically extracted — no manual reading needed:

People / Persons [PERSON]:
    → Kamau Thugge

Organisations [ORG]:
    → The Central Bank of Kenya Annual Report
    → Cabinet
    → the National
Treasury
    → The Monetary Policy Committee
    → the Central
Bank Rate
    → JPMorgan Chase Bank of the United States
    → a Representative Office
    → CBK
    → MAC
    → IMF
    → World Bank

Countries / Cities [GPE]:
    → Nairobi
    → Kenya
    → Mombasa

Dates [DAT

In [ ]:
# 4 — Sentence Labeling Results
# Full sentence-by-sentence labeling from the CBK Report PDF text

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

# Use cbk_text loaded in Cell 1
sentences = sent_tokenize(cbk_text)

tag_meanings = {
    'NN':'Noun',        'NNS':'Noun(pl)',     'NNP':'Proper Noun',
    'VBD':'Verb(past)', 'VBZ':'Verb(pres)',   'VBN':'Verb(pp)',
    'JJ':'Adjective',   'RB':'Adverb',        'DT':'Determiner',
    'IN':'Preposition', 'CD':'Number',        'CC':'Conjunction',
}

print("=" * 60)
print("  4 — SENTENCE LABELING RESULTS")
print("  Dataset: CBK Annual Report & Financial Statements 2024/25")
print("=" * 60)
print(f"\nTotal sentences extracted from CBK Report: {len(sentences)}\n")

# Label and display first 5 sentences
for i, sent in enumerate(sentences[:5], 1):
    tokens = word_tokenize(sent)
    tagged = nltk.pos_tag(tokens)

    # Summarise what was found
    from collections import Counter
    tag_counts = Counter(tag for _, tag in tagged)
    proper_nouns = [w for w, t in tagged if t == 'NNP']
    numbers      = [w for w, t in tagged if t == 'CD']

    print(f"Sentence {i}:")
    print(f"  Text         : {sent.strip()[:100]}")
    print(f"  Word count   : {len(tokens)}")
    print(f"  Proper Nouns : {proper_nouns if proper_nouns else 'none'}")
    print(f"  Numbers/Pct  : {numbers if numbers else 'none'}")
    print(f"  All labels   : {tagged}")
    print()

print("=" * 60)
print("LABELING SUMMARY:")
print(f"  Each word gets one sequence label.")
print(f"  Proper Nouns (NNP) = names, places, organisations.")
print(f"  Numbers (CD)       = financial figures, percentages.")
print(f"  This is sequence labeling applied to real CBK data.")
print("=" * 60)

  4 — SENTENCE LABELING RESULTS
  Dataset: CBK Annual Report & Financial Statements 2024/25

Total sentences extracted from CBK Report: 38

Sentence 1:
  Text         : IICENTRAL BANK OF KENYA
ANNUAL REPORT & FINANCIAL STATEMENTS 2024/25
To be a World Class Modern Cent
  Word count   : 96
  Proper Nouns : ['IICENTRAL', 'BANK', 'OF', 'KENYA', 'ANNUAL', 'REPORT', 'FINANCIAL', 'STATEMENTS', 'World', 'Class', 'Modern', 'Central', 'Bank', 'IICENTRAL', 'BANK', 'OF', 'KENYA', 'ANNUAL', 'REPORT', 'FINANCIAL', 'STATEMENTS', 'World', 'Class', 'Modern', 'Central', 'Bank', 'World', 'Class', 'Modern', 'Central', 'BankCENTRAL', 'BANK', 'OF', 'KENYA', 'ANNUAL', 'REPORT', 'FINANCIAL', 'STATEMENTS', 'CBK', 'WINS', 'GLOBAL', 'BANKNOTE', 'AWARD', 'Central', 'Bank', 'Kenya', 'Best', 'New', 'Series', 'Banknotes', 'High', 'Security', 'Printing', 'Europe', 'Middle', 'East', 'Africa', 'HSP-EMEA', 'Basel', 'Switzerland', 'February']
  Numbers/Pct  : ['2024/25', '2024/25To', '2024/25III', '4', '2025']
  All lab

In [ ]:
# 5 — Dataset Used for Training or Testing
# Showing the CBK Annual Report 2024/25 as the NLP dataset

import PyPDF2
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from collections import Counter

pdf_path = '/content/1084981846_2025 Annual Report.pdf'

# Full dataset stats
with open(pdf_path, 'rb') as f:
    reader = PyPDF2.PdfReader(f)
    total_pages = len(reader.pages)

    # Extract broader range for dataset overview
    full_text = ''
    for page_num in range(7, 40):
        page_text = reader.pages[page_num].extract_text()
        if page_text:
            full_text += page_text + ' '

stop_words   = set(stopwords.words('english'))
all_tokens   = word_tokenize(full_text)
word_tokens  = [w for w in all_tokens if w.isalpha()]
clean_tokens = [w.lower() for w in word_tokens if w.lower() not in stop_words]
sentences    = sent_tokenize(full_text)
freq         = Counter(clean_tokens)

print("=" * 60)
print("  5 — DATASET USED FOR TRAINING AND TESTING")
print("=" * 60)
print(f"\n  Document   : CBK Annual Report & Financial Statements")
print(f"  Year       : 2024/25")
print(f"  Publisher  : Central Bank of Kenya")
print(f"  Source     : www.centralbank.go.ke")
print(f"  File       : 1084981846_2025_Annual_Report.pdf")
print(f"\n  DATASET STATISTICS:")
print(f"  {'Total pages in document':<35}: {total_pages}")
print(f"  {'Pages used for NLP tasks':<35}: 33 (pages 7–40)")
print(f"  {'Total characters extracted':<35}: {len(full_text):,}")
print(f"  {'Total sentences':<35}: {len(sentences):,}")
print(f"  {'Total word tokens':<35}: {len(word_tokens):,}")
print(f"  {'Clean tokens (no stopwords)':<35}: {len(clean_tokens):,}")
print(f"  {'Unique words':<35}: {len(freq):,}")
print(f"\n  TOP 10 KEYWORDS IN DATASET:")
for word, count in freq.most_common(10):
    print(f"    {word:<22} {count}")
print(f"\n  SAMPLE TEXT FROM DATASET (first 300 chars):")
print(f"  {full_text[:300].strip()}")
print("\n" + "=" * 60)
print("This real CBK document is the dataset for all NLP tasks.")
print("It contains financial language, named entities, statistics")
print("and policy language — ideal for NLP training and testing.")

  5 — DATASET USED FOR TRAINING AND TESTING

  Document   : CBK Annual Report & Financial Statements
  Year       : 2024/25
  Publisher  : Central Bank of Kenya
  Source     : www.centralbank.go.ke
  File       : 1084981846_2025_Annual_Report.pdf

  DATASET STATISTICS:
  Total pages in document            : 152
  Pages used for NLP tasks           : 33 (pages 7–40)
  Total characters extracted         : 70,923
  Total sentences                    : 270
  Total word tokens                  : 8,646
  Clean tokens (no stopwords)        : 5,837
  Unique words                       : 1,289

  TOP 10 KEYWORDS IN DATASET:
    e                      177
    percent                158
    financial              99
    bank                   85
    r                      83
    n                      82
    central                67
    kenya                  63
    c                      56
    growth                 55

  SAMPLE TEXT FROM DATASET (first 300 chars):
  VICENTRAL BANK OF KENYA
AN